# Gradient Boosting Regressor — Thermal Predictor

Modelo de gradient boosting que prevê a temperatura de cada termopar (`tc_name`) de
uma `key_area` a partir das sete variáveis de configuração do ensaio.

**Um modelo por setor:** cada `key_area` recebe seu próprio modelo, treinado apenas
com os runs e os termopares daquela região — coerente com a H1 (Seção 4.2.4), em que
o efeito de uma configuração muda de sinal conforme a área.

**Entrada:** planilhas em formato wide geradas por `tratamento_final_dos_dados.ipynb`
(`data/<key_area>_data.csv`) — uma linha por run, uma coluna por termopar.

## Diferença estrutural em relação aos outros dois notebooks

`RandomForestRegressor` e `DecisionTreeRegressor` aceitam `Y` matricial nativamente:
uma folha guarda um vetor de médias, uma por termopar. **`GradientBoostingRegressor`
não** — ele exige `y` unidimensional e falha com
`ValueError: y should be a 1d array` se receber vários alvos.

A solução aqui é `MultiOutputRegressor`, que treina **um GBR independente por
termopar** e junta as previsões. Duas consequências práticas:

1. Os hiperparâmetros da grade levam o prefixo `estimator__`, porque são repassados
   ao GBR de dentro do wrapper.
2. Há uma importância de variáveis **por termopar**, e não uma única para o modelo
   inteiro — o que dá mais informação que o Random Forest, e é explorado na Célula 5.

Fora isso, a preparação dos dados segue a do `Random_for_regre.ipynb`, com duas
diferenças deliberadas: a codificação é one-hot (Célula 3) e o R² de cada área é
resumido pela **mediana** entre termopares, e não pela média (Célula 6).

**Estrutura:**

- **Seções 2 a 6** — analisam **uma** área em detalhe (altere `CAMINHO_PLANILHA` na
  Célula 2 para trocar de área): leitura, codificação, divisão, treino com
  `GridSearchCV` e métricas por termopar.
- **Seção 7** — treina as **sete** áreas de uma vez no corte 70/20/10 (holdout).
- **Seção 8** — avaliação oficial por LOGO, com os gráficos por área e a
  confirmação dos hiperparâmetros por busca em grade avaliada por LOGO.
- **Seção 9** — explicabilidade: importância por impureza e SHAP.

## 1. Bibliotecas e configuração global

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV, LeaveOneGroupOut
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
SEMENTE = 42

## 2. Parâmetros e leitura da planilha

Formato wide: cada termopar é uma coluna-alvo independente. Tudo que não está em
`VARIAVEIS_INDEPENDENTES` é candidato a alvo.

Duas checagens são feitas na leitura, porque nem toda coluna extra é um termopar
válido: colunas **sem nome** (`Unnamed: N`, geradas quando o `tc_name` original
estava em branco — é o caso de `ct_knobs_data.csv`) e colunas **inteiramente
vazias** são descartadas com aviso, em vez de entrarem como alvo do modelo.

In [ ]:
# ============================================================
# CÉLULA 2 — PARÂMETROS E LEITURA DA PLANILHA
# Formato wide: cada termopar é uma coluna-alvo independente
# ============================================================

# Troque aqui para analisar outra key_area em detalhe:
#   ../data/mainback_data.csv        ../data/ipc_comp_data.csv
#   ../data/coil_temps_data.csv      ../data/ipc_cooling_data.csv
#   ../data/main_vent_sys_data.csv   ../data/oven_hmi_data.csv
#   ../data/ct_knobs_data.csv
CAMINHO_PLANILHA = "../data/mainback_data.csv"

VARIAVEIS_INDEPENDENTES = [
    "blower_direction",
    "housing",
    "board_position",
    "run_case",
    "losses",
    "deflector_hole",
    "material_coil_support",
]

# Um termopar só entra como alvo se tiver leitura em pelo menos esta fração dos
# runs. Sensores abaixo disso são removidos como COLUNA — descartá-los custa um
# alvo, enquanto exigir que estejam presentes custaria quase todos os runs.
COBERTURA_MINIMA_ALVO = 0.50

# Número de folds do GridSearchCV. É reduzido automaticamente se o conjunto de
# treino for pequeno demais (ver Célula 5).
N_FOLDS_DESEJADO = 3


def identificar_alvos(df, independentes):
    """Separa as colunas-alvo válidas das colunas que não representam termopar."""
    alvos, descartadas = [], []
    for coluna in df.columns:
        if coluna in independentes:
            continue
        if str(coluna).startswith("Unnamed:") or not str(coluna).strip():
            descartadas.append((coluna, "coluna sem nome de termopar"))
        elif df[coluna].isna().all():
            descartadas.append((coluna, "coluna inteiramente vazia"))
        else:
            alvos.append(coluna)
    return alvos, descartadas


def resumir_alvos(df, alvos):
    """Tabela de cobertura e estatística descritiva de cada termopar."""
    return pd.DataFrame([
        {
            "termopar": alvo,
            "leituras": int(df[alvo].notna().sum()),
            "cobertura": df[alvo].notna().mean(),
            "min": df[alvo].min(),
            "max": df[alvo].max(),
            "média": df[alvo].mean(),
            "desvio": df[alvo].std(),
        }
        for alvo in alvos
    ]).sort_values("cobertura", ascending=False, ignore_index=True)


def listar_alvos_duplicados(df, alvos):
    """Pares de termopares com leitura idêntica (possível duplicação de sensor)."""
    return [
        (alvo_a, alvo_b)
        for i, alvo_a in enumerate(alvos)
        for alvo_b in alvos[i + 1:]
        if df[alvo_a].equals(df[alvo_b])
    ]


dados = pd.read_csv(CAMINHO_PLANILHA)
VARIAVEIS_ALVO, COLUNAS_DESCARTADAS = identificar_alvos(dados, VARIAVEIS_INDEPENDENTES)

print(f"Arquivo: {CAMINHO_PLANILHA}")
print(f"Linhas: {len(dados)} | Colunas: {dados.shape[1]}")

if COLUNAS_DESCARTADAS:
    print("\nColunas ignoradas (não são termopares válidos):")
    for coluna, motivo in COLUNAS_DESCARTADAS:
        print(f"   {coluna!r}: {motivo}")

print(f"\nTermopares candidatos ({len(VARIAVEIS_ALVO)}):")
print(resumir_alvos(dados, VARIAVEIS_ALVO).round(2).to_string(index=False))

print(f"\nValores nulos nos alvos: {int(dados[VARIAVEIS_ALVO].isna().sum().sum())}")

duplicadas = listar_alvos_duplicados(dados, VARIAVEIS_ALVO)
if duplicadas:
    print("\nATENÇÃO — colunas-alvo com valores idênticos:")
    for alvo_a, alvo_b in duplicadas:
        print(f"   {alvo_a}  ==  {alvo_b}")

dados.head()

## 3. Seleção de alvos, padronização das categorias e codificação

Primeiro descarta o **sensor** de cobertura baixa (coluna), depois descarta as
**linhas** que ainda tenham buraco entre os alvos que sobraram. A ordem importa:
fazer `dropna` direto sobre todos os alvos é *listwise deletion* — basta um sensor
pouco medido para eliminar quase todos os runs. Em `main_vent_sys_data.csv`, dois
sensores medidos em apenas 4 dos 56 runs derrubariam a base de 56 para 4 linhas.

Idêntico ao `Random_for_regre.ipynb`, de propósito: as três abordagens precisam
partir exatamente da mesma base para que a comparação de métricas faça sentido.

In [ ]:
# ============================================================
# CÉLULA 3 — SELEÇÃO DE ALVOS, PADRONIZAÇÃO E CODIFICAÇÃO ONE-HOT
# ============================================================

def filtrar_alvos_por_cobertura(df, alvos, cobertura_minima):
    """Mantém apenas os termopares medidos em uma fração mínima dos runs."""
    mantidos, removidos = [], []
    for alvo in alvos:
        cobertura = df[alvo].notna().mean()
        if cobertura >= cobertura_minima:
            mantidos.append(alvo)
        else:
            removidos.append((alvo, int(df[alvo].notna().sum()), cobertura))
    return mantidos, removidos


def padronizar_categorias(df, independentes):
    """Transforma nulos em categoria explícita. As grafias divergentes não são mais
    corrigidas aqui: já chegam padronizadas de `tratamento_final_dos_dados.ipynb`."""
    df = df.copy()
    for variavel in independentes:
        df[variavel] = df[variavel].fillna("Nao Informado").astype(str)
    return df


def codificar_onehot(df, independentes):
    """Codificação one-hot: uma coluna binária por categoria observada."""
    X = pd.get_dummies(df[independentes].astype(str), columns=independentes).astype(int)
    dicionario = {
        variavel: sorted(df[variavel].astype(str).unique())
        for variavel in independentes
    }
    return X, dicionario


# 1) Descarta o SENSOR pouco medido (coluna), não o run (linha).
VARIAVEIS_ALVO, alvos_removidos = filtrar_alvos_por_cobertura(
    dados, VARIAVEIS_ALVO, COBERTURA_MINIMA_ALVO
)

if alvos_removidos:
    print(f"Termopares removidos por cobertura < {COBERTURA_MINIMA_ALVO:.0%}:")
    for alvo, leituras, cobertura in alvos_removidos:
        print(f"   {alvo:40s} {leituras:3d}/{len(dados)} runs ({cobertura:.1%})")
    print()

if not VARIAVEIS_ALVO:
    raise ValueError(
        f"Nenhum termopar atingiu a cobertura mínima de {COBERTURA_MINIMA_ALVO:.0%}. "
        f"Reduza COBERTURA_MINIMA_ALVO na Célula 2 para modelar este arquivo."
    )

# 2) Só agora descarta as LINHAS com buraco entre os alvos que sobraram.
dados_tratados = dados.dropna(subset=VARIAVEIS_ALVO).copy()

linhas_removidas = len(dados) - len(dados_tratados)
if linhas_removidas:
    print(f"Linhas removidas por alvo ausente: {linhas_removidas}\n")

dados_tratados = padronizar_categorias(
    dados_tratados, VARIAVEIS_INDEPENDENTES
)
X, dicionario_codificacao = codificar_onehot(dados_tratados, VARIAVEIS_INDEPENDENTES)
Y = dados_tratados[VARIAVEIS_ALVO]

print("Categorias por variável (uma coluna one-hot por categoria):\n")
for variavel, categorias in dicionario_codificacao.items():
    print(f"{variavel} ({len(categorias)} categorias)")
    for categoria in categorias:
        print(f"   {variavel}_{categoria}")
    print()

print(f"Matriz de features: {X.shape}")
print(f"Colunas-alvo: {len(VARIAVEIS_ALVO)}")
print(f"Linhas úteis: {len(dados_tratados)} de {len(dados)}")
print(f"Combinações distintas de configuração: {X.drop_duplicates().shape[0]}")

## 4. Separação treino (70%) / teste (20%) / verificação (10%)

`Y` é matricial: uma coluna por termopar, lida de `dados_tratados` para permanecer
alinhada com as linhas de `X`. O `MultiOutputRegressor` da Célula 5 se encarrega de
quebrar essa matriz em um problema por termopar.

A célula checa antes da divisão se sobraram linhas suficientes para formar os três
conjuntos e ainda alimentar a validação cruzada, falhando com uma mensagem que diz
o que ajustar — em vez do `ValueError` genérico do `GridSearchCV`.

In [ ]:
# ============================================================
# CÉLULA 4 — SEPARAÇÃO TREINO (70%) / TESTE (20%) / VERIFICAÇÃO (10%)
# Y é matricial (uma coluna por termopar)
# ============================================================
PROPORCAO_TREINO = 0.70
PROPORCAO_TESTE = 0.20
PROPORCAO_VERIFICACAO = 0.10

MINIMO_DE_LINHAS = 10
if len(X) < MINIMO_DE_LINHAS:
    raise ValueError(
        f"Apenas {len(X)} linhas úteis após o tratamento — insuficiente para dividir em "
        f"treino/teste/verificação e rodar validação cruzada. Verifique a Célula 3: "
        f"provavelmente há termopares esparsos derrubando linhas. Considere reduzir "
        f"COBERTURA_MINIMA_ALVO (hoje {COBERTURA_MINIMA_ALVO:.0%}) na Célula 2."
    )

X_treino, X_restante, Y_treino, Y_restante = train_test_split(
    X, Y,
    train_size=PROPORCAO_TREINO,
    random_state=SEMENTE,
)

proporcao_verificacao_no_restante = (
    PROPORCAO_VERIFICACAO / (PROPORCAO_TESTE + PROPORCAO_VERIFICACAO)
)

X_teste, X_verificacao, Y_teste, Y_verificacao = train_test_split(
    X_restante, Y_restante,
    test_size=proporcao_verificacao_no_restante,
    random_state=SEMENTE,
)

print(f"Treino:       {len(X_treino):3d} linhas ({len(X_treino)/len(X):.1%})")
print(f"Teste:        {len(X_teste):3d} linhas ({len(X_teste)/len(X):.1%})")
print(f"Verificação:  {len(X_verificacao):3d} linhas ({len(X_verificacao)/len(X):.1%})")
print(f"\nFormato de X: {X_treino.shape}  |  Formato de Y: {Y_treino.shape}")
print(f"Modelos que serão treinados nesta área: {Y_treino.shape[1]} "
      f"(um GBR por termopar)")

## 5. Treinamento do modelo (Gradient Boosting, um GBR por termopar)

`MultiOutputRegressor` envolve o `GradientBoostingRegressor` e treina um regressor
independente por termopar — é o que torna o GBR utilizável neste dataset multi-alvo.
Por isso os parâmetros da grade levam o prefixo `estimator__`.

A grade é a de controle de superajuste que interessa em boosting: `learning_rate` e
`n_estimators` (que se compensam — taxa menor exige mais estágios), `max_depth`
raso (2 ou 3, o padrão em boosting é somar muitos aprendizes fracos, não poucos
fortes) e `subsample` < 1 para introduzir estocasticidade, útil com as ~22 linhas de treino de cada área.

`n_jobs` fica no `GridSearchCV`, e o `MultiOutputRegressor` roda em série, para não
disputar núcleos entre os dois níveis de paralelismo.

A grade inclui `learning_rate = 0,03`, o valor usado na avaliação por LOGO (Seção
8), para que a configuração oficial esteja dentro do espaço buscado. São 24
combinações (2 × 3 × 2 × 2).

**Custo:** 24 combinações × 3 folds × um GBR por termopar, em poucos segundos por
área.

In [ ]:
# ============================================================
# CÉLULA 5 — TREINAMENTO (GRADIENT BOOSTING, UM GBR POR TERMOPAR)
# GradientBoostingRegressor não aceita Y matricial: MultiOutputRegressor
# treina um regressor independente para cada termopar.
# ============================================================
GRADE_HIPERPARAMETROS = {
    "estimator__n_estimators":  [100, 300],
    "estimator__learning_rate": [0.03, 0.05, 0.1],
    "estimator__max_depth":     [2, 3],
    "estimator__subsample":     [1.0, 0.8],
}

# Nunca pedir mais folds do que há linhas de treino.
n_folds = max(2, min(N_FOLDS_DESEJADO, len(X_treino)))
if n_folds < N_FOLDS_DESEJADO:
    print(f"Aviso: {len(X_treino)} linhas de treino — usando {n_folds} folds "
          f"em vez de {N_FOLDS_DESEJADO}.\n")

busca = GridSearchCV(
    estimator=MultiOutputRegressor(
        GradientBoostingRegressor(random_state=SEMENTE), n_jobs=1
    ),
    param_grid=GRADE_HIPERPARAMETROS,
    cv=n_folds,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
busca.fit(X_treino, Y_treino)

modelo = busca.best_estimator_

print("Melhores hiperparâmetros:")
for parametro, valor in busca.best_params_.items():
    print(f"   {parametro.replace('estimator__', '')}: {valor}")

print(f"\nRegressores treinados: {len(modelo.estimators_)} (um por termopar)")


def importancias_por_alvo(modelo_multioutput, colunas_onehot, independentes, alvos):
    """Matriz termopar x variável — soma a importância das colunas one-hot de
    volta para a variável original (cada categoria virou uma coluna binária)."""
    bruta = pd.DataFrame(
        [estimador.feature_importances_ for estimador in modelo_multioutput.estimators_],
        index=alvos,
        columns=colunas_onehot,
    )

    def variavel_original(coluna):
        candidatos = [v for v in independentes if coluna.startswith(v + "_")]
        return max(candidatos, key=len)

    mapa_variavel = {coluna: variavel_original(coluna) for coluna in colunas_onehot}
    return bruta.T.groupby(mapa_variavel).sum().T


matriz_importancia = importancias_por_alvo(modelo, X.columns, VARIAVEIS_INDEPENDENTES, VARIAVEIS_ALVO)
importancia = matriz_importancia.mean().sort_values(ascending=False)

print(f"\nImportância média das variáveis (entre os {len(VARIAVEIS_ALVO)} termopares):")
print(importancia.round(4).to_string())

print("\nTermopar mais sensível a cada variável:")
for variavel in importancia.index:
    alvo = matriz_importancia[variavel].idxmax()
    print(f"   {variavel:24s} -> {alvo} ({matriz_importancia.loc[alvo, variavel]:.3f})")

## 6. Avaliação estatística por termopar

MAE, MSE, RMSE, R² e MAPE calculados coluna a coluna nos três conjuntos. Para
resumir cada conjunto em um número, MAE, MSE, RMSE e MAPE usam a **média** entre
termopares e o R² usa a **mediana**: o R² de um termopar quase constante no
conjunto avaliado tem denominador perto de zero e pode chegar a centenas negativas,
o que dominaria uma média sem dizer nada sobre os demais sensores. A mesma
agregação é usada no holdout (Seções 6 e 7) e no LOGO (Seção 8).

Fecha com a fração de previsões dentro da margem de 10% acordada com o parceiro e o
gap de MAE entre treino e teste.

In [ ]:
# ============================================================
# CÉLULA 6 — AVALIAÇÃO ESTATÍSTICA POR TERMOPAR
# ============================================================
def avaliar_multioutput(Y_real, Y_previsto, nome_conjunto, alvos):
    """Calcula métricas para cada coluna-alvo separadamente."""
    linhas = []
    for indice, termopar in enumerate(alvos):
        real = Y_real.iloc[:, indice].values
        previsto = Y_previsto[:, indice]
        mse = mean_squared_error(real, previsto)
        linhas.append({
            "conjunto": nome_conjunto,
            "termopar": termopar,
            "MAE":      mean_absolute_error(real, previsto),
            "MSE":      mse,
            "RMSE":     np.sqrt(mse),
            "R2":       r2_score(real, previsto),
            "MAPE_%":   np.mean(np.abs((real - previsto) / real)) * 100,
        })
    return linhas


previsoes_treino      = modelo.predict(X_treino)
previsoes_teste       = modelo.predict(X_teste)
previsoes_verificacao = modelo.predict(X_verificacao)

avaliacao = pd.DataFrame(
    avaliar_multioutput(Y_treino,      previsoes_treino,      "Treino",      VARIAVEIS_ALVO)
  + avaliar_multioutput(Y_teste,       previsoes_teste,       "Teste",       VARIAVEIS_ALVO)
  + avaliar_multioutput(Y_verificacao, previsoes_verificacao, "Verificação", VARIAVEIS_ALVO)
)

print("=" * 100)
print(f"DESEMPENHO POR TERMOPAR — {CAMINHO_PLANILHA}")
print("=" * 100)
print(avaliacao.round(4).to_string(index=False))

print("\n" + "=" * 100)
print("RESUMO ENTRE OS TERMOPARES (R2: mediana; demais: média)")
print("=" * 100)
# R² resumido pela mediana entre termopares; as demais métricas, pela média.
AGREGACAO_METRICAS = {"MAE": "mean", "MSE": "mean", "RMSE": "mean",
                      "R2": "median", "MAPE_%": "mean"}
resumo_conjuntos = avaliacao.groupby("conjunto", sort=False).agg(AGREGACAO_METRICAS)
print(resumo_conjuntos.round(4).to_string())

erro_percentual_teste = np.abs((Y_teste.values - previsoes_teste) / Y_teste.values)
print(f"\nPrevisões dentro da margem de 10% (teste, todos os termopares): "
      f"{(erro_percentual_teste <= 0.10).mean():.1%}")

# Gap treino -> teste: leitura direta de superajuste (risco R03 da Seção 4.1.5).
gap = resumo_conjuntos.loc["Teste", "MAE"] - resumo_conjuntos.loc["Treino", "MAE"]
print(f"Gap de MAE treino -> teste: {gap:+.2f} °C")

## 7. Um modelo por setor — treino das sete áreas

Encadeia as funções das células anteriores em um único pipeline e treina um conjunto
independente de GBRs para cada `key_area`, com o mesmo `GridSearchCV` da Célula 5.
Os modelos ficam guardados em `MODELOS_POR_AREA`, junto com os alvos e o
dicionário de codificação de cada um, e são reaproveitados na explicabilidade
(Seção 9).

Ao todo, um GBR para cada termopar de cada área (61 nas sete planilhas). A tabela
ao final é o resultado no corte 70/20/10 (holdout) e serve de **referência**: a
avaliação oficial é a da Seção 8.

In [ ]:
# ============================================================
# CÉLULA 7 — UM MODELO POR SETOR (KEY_AREA)
# ============================================================
PLANILHAS = [
    "../data/mainback_data.csv",
    "../data/coil_temps_data.csv",
    "../data/main_vent_sys_data.csv",
    "../data/ct_knobs_data.csv",
    "../data/ipc_comp_data.csv",
    "../data/ipc_cooling_data.csv",
    "../data/oven_hmi_data.csv",
]


def preparar_area(caminho, cobertura_minima=COBERTURA_MINIMA_ALVO):
    """Leitura + seleção de alvos + codificação de uma planilha. Devolve X, Y e metadados."""
    df = pd.read_csv(caminho)
    alvos, _ = identificar_alvos(df, VARIAVEIS_INDEPENDENTES)
    alvos, removidos = filtrar_alvos_por_cobertura(df, alvos, cobertura_minima)
    if not alvos:
        raise ValueError(f"{caminho}: nenhum termopar com cobertura >= {cobertura_minima:.0%}")

    tratado = df.dropna(subset=alvos)
    tratado = padronizar_categorias(tratado, VARIAVEIS_INDEPENDENTES)
    X_area, dicionario = codificar_onehot(tratado, VARIAVEIS_INDEPENDENTES)
    return X_area, tratado[alvos], alvos, dicionario, removidos


def treinar_area(caminho, n_folds_desejado=N_FOLDS_DESEJADO):
    """Treina os GBRs de uma key_area e devolve o modelo com suas métricas."""
    X_area, Y_area, alvos, dicionario, removidos = preparar_area(caminho)

    if len(X_area) < MINIMO_DE_LINHAS:
        raise ValueError(f"{caminho}: apenas {len(X_area)} linhas úteis")

    X_tr, X_resto, Y_tr, Y_resto = train_test_split(
        X_area, Y_area, train_size=PROPORCAO_TREINO, random_state=SEMENTE)
    X_te, X_ve, Y_te, Y_ve = train_test_split(
        X_resto, Y_resto,
        test_size=PROPORCAO_VERIFICACAO / (PROPORCAO_TESTE + PROPORCAO_VERIFICACAO),
        random_state=SEMENTE)

    modelo_area = GridSearchCV(
        estimator=MultiOutputRegressor(
            GradientBoostingRegressor(random_state=SEMENTE), n_jobs=1
        ),
        param_grid=GRADE_HIPERPARAMETROS,
        cv=max(2, min(n_folds_desejado, len(X_tr))),
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
    ).fit(X_tr, Y_tr).best_estimator_

    metricas = pd.DataFrame(
        avaliar_multioutput(Y_tr, modelo_area.predict(X_tr), "Treino", alvos)
      + avaliar_multioutput(Y_te, modelo_area.predict(X_te), "Teste", alvos)
      + avaliar_multioutput(Y_ve, modelo_area.predict(X_ve), "Verificação", alvos)
    )
    medias = metricas.groupby("conjunto", sort=False).agg(AGREGACAO_METRICAS)
    erro_pct = np.abs((Y_te.values - modelo_area.predict(X_te)) / Y_te.values)

    resumo = {
        "setor": caminho.split("/")[-1].replace("_data.csv", ""),
        "linhas": len(X_area),
        "alvos": len(alvos),
        "alvos_removidos": len(removidos),
        "MAE_treino": medias.loc["Treino", "MAE"],
        "MAE_teste": medias.loc["Teste", "MAE"],
        "RMSE_teste": medias.loc["Teste", "RMSE"],
        "R2_teste_mediana": medias.loc["Teste", "R2"],
        "MAPE_teste_%": medias.loc["Teste", "MAPE_%"],
        "dentro_de_10%": (erro_pct <= 0.10).mean(),
    }
    detalhe = {
        "modelo": modelo_area,
        "alvos": alvos,
        "codificacao": dicionario,
        "metricas": metricas,
        "importancias": importancias_por_alvo(modelo_area, X_area.columns, VARIAVEIS_INDEPENDENTES, alvos),
    }
    return resumo, detalhe


MODELOS_POR_AREA = {}
resumos = []

for caminho in PLANILHAS:
    try:
        resumo, detalhe = treinar_area(caminho)
        MODELOS_POR_AREA[resumo["setor"]] = detalhe
        resumos.append(resumo)
        print(f"OK    {resumo['setor']:15s} "
              f"{resumo['linhas']:3d} linhas, {resumo['alvos']:2d} GBRs treinados")
    except Exception as erro:
        print(f"FALHA {caminho}: {type(erro).__name__}: {erro}")

total_gbrs = sum(r["alvos"] for r in resumos)
print(f"\n{len(MODELOS_POR_AREA)} setores modelados, {total_gbrs} GBRs no total: "
      f"{list(MODELOS_POR_AREA)}")

comparativo_holdout = pd.DataFrame(resumos)

print("\n" + "=" * 125)
print("COMPARATIVO ENTRE SETORES (HOLDOUT 70/20/10) — referência; a avaliação oficial é o LOGO da Seção 8")
print("=" * 125)
print(comparativo_holdout.round(4).to_string(index=False))

print("\n" + "=" * 125)
print("MÉDIA ENTRE OS SETORES (HOLDOUT)")
print("=" * 125)
print(comparativo_holdout[["MAE_treino", "MAE_teste", "RMSE_teste", "R2_teste_mediana",
                           "MAPE_teste_%", "dentro_de_10%"]].mean().round(4).to_string())

## 8. Avaliação oficial via LOGO (Leave-One-Group-Out)

Um split único 70/20/10 deixa só ~6 linhas no conjunto de teste de cada área, de
32 configurações totais. Métricas calculadas sobre tão poucas linhas são instáveis
(o R² em particular, cujo denominador é a variância do próprio conjunto de teste,
pode explodir) — por isso a tabela da Seção 7 é só referência, e a validação
oficial é o LOGO.

**LOGO (Leave-One-Group-Out)**, com grupo = configuração de ensaio: como cada uma
das 32 linhas de uma área é uma configuração distinta, equivale a Leave-One-Out —
cada configuração vira teste uma vez, treinando com as outras 31. Toda a base é
usada como teste (em rodadas diferentes), sem depender da sorte de um split.

**Hiperparâmetros fixos** (`PARAMS_LOGO`), iguais nas sete áreas e não re-tunados
por rodada: `n_estimators = 100`, `learning_rate = 0,03`, `max_depth = 3`,
`subsample = 0,8`. Foram definidos nos testes do grupo com o Gradient Boosting e
são confirmados por uma busca em grade avaliada pelo próprio LOGO na Seção 8.2.
Rodar `GridSearchCV` dentro de cada uma das 32 × 7 rodadas seria caro, e a Seção 10
do `Random_for_regre.ipynb` já mostrou que, com poucos dados, a combinação
vencedora do Grid Search fica empatada dentro do ruído com várias outras.

Métricas agregadas como na Seção 6: MAE, RMSE e MAPE calculados por termopar e
resumidos pela média; R² por termopar resumido pela mediana.

In [ ]:
# ============================================================
# CÉLULA 8 — AVALIAÇÃO OFICIAL VIA LOGO (LEAVE-ONE-GROUP-OUT)
# Reusa as funções e variáveis já definidas (identificar_alvos,
# filtrar_alvos_por_cobertura, padronizar_categorias, codificar_onehot,
# PLANILHAS, VARIAVEIS_INDEPENDENTES, SEMENTE).
# ============================================================
# Hiperparametros oficiais, fixos entre areas (confirmados na Secao 8.2).
PARAMS_LOGO = dict(n_estimators=100, learning_rate=0.03, max_depth=3,
                    subsample=0.8, random_state=SEMENTE)


def avaliar_logo(caminho, cobertura_minima=COBERTURA_MINIMA_ALVO):
    """Leave-One-Out sobre as configurações de uma planilha (cada linha já
    é uma configuração única, então LeaveOneGroupOut == LeaveOneOut)."""
    df = pd.read_csv(caminho)
    alvos, _ = identificar_alvos(df, VARIAVEIS_INDEPENDENTES)
    alvos, _ = filtrar_alvos_por_cobertura(df, alvos, cobertura_minima)
    tratado = df.dropna(subset=alvos).copy()
    tratado = padronizar_categorias(tratado, VARIAVEIS_INDEPENDENTES)
    X, _ = codificar_onehot(tratado, VARIAVEIS_INDEPENDENTES)
    Y = tratado[alvos]

    n = len(X)
    pred = np.zeros(Y.shape)
    for i in range(n):
        treino = np.arange(n) != i
        modelo = MultiOutputRegressor(GradientBoostingRegressor(**PARAMS_LOGO), n_jobs=1)
        modelo.fit(X.values[treino], Y.values[treino])
        pred[i] = modelo.predict(X.values[[i]])[0]

    real = Y.values
    erro_abs = np.abs(real - pred)
    mae = erro_abs.mean()
    # RMSE por termopar e depois a media, como no holdout (Secao 6).
    rmse = float(np.mean(np.sqrt(((real - pred) ** 2).mean(axis=0))))
    r2_por_sensor = []
    for j in range(real.shape[1]):
        y_j, p_j = real[:, j], pred[:, j]
        ss_res = ((y_j - p_j) ** 2).sum()
        ss_tot = ((y_j - y_j.mean()) ** 2).sum()
        r2_por_sensor.append(1 - ss_res / ss_tot if ss_tot > 0 else np.nan)
    r2_mediana = float(np.nanmedian(r2_por_sensor))
    mape = np.mean(np.abs((real - pred) / real)) * 100
    erro_pct = np.abs((real - pred) / real)
    return {
        "setor": caminho.split("/")[-1].replace("_data.csv", ""),
        "n_configs": n, "alvos": len(alvos),
        "MAE_logo": mae, "RMSE_logo": rmse, "R2_logo_mediana": r2_mediana,
        "MAPE_logo_%": mape, "dentro_de_10%_logo": (erro_pct <= 0.10).mean(),
    }


resultados_logo = []
for caminho in PLANILHAS:
    try:
        resultados_logo.append(avaliar_logo(caminho))
        print(f"OK    {resultados_logo[-1]['setor']:15s} "
              f"LOGO concluído ({resultados_logo[-1]['n_configs']} configs)")
    except Exception as erro:
        print(f"FALHA {caminho}: {type(erro).__name__}: {erro}")

comparativo_logo = pd.DataFrame(resultados_logo)

print("\n" + "=" * 125)
print("COMPARATIVO ENTRE SETORES (LOGO) — Gradient Boosting, avaliação Leave-One-Out")
print("=" * 125)
print(comparativo_logo.round(4).to_string(index=False))

print("\n" + "=" * 125)
print("MÉDIA ENTRE OS SETORES (LOGO) — oficial")
print("=" * 125)
print(comparativo_logo[["MAE_logo", "RMSE_logo", "R2_logo_mediana", "MAPE_logo_%", "dentro_de_10%_logo"]]
      .mean().round(4).to_string())

### 8.1. Métricas da validação LOGO, por área

A tabela acima já traz os números; os três gráficos abaixo servem para
comparar rapidamente as sete áreas entre si, na mesma ordem (por MAE
crescente): erro absoluto, poder de distinção (R², mediana entre os
termopares) e fração de previsões dentro da margem de 10% acordada com o
parceiro.


In [ ]:
# ------------------------------------------------------------
# Grafico -- metricas da validacao LOGO, por area
# ------------------------------------------------------------
ORDEM_AREAS = comparativo_logo.sort_values("MAE_logo")["setor"].tolist()
tabela_ord = comparativo_logo.set_index("setor").loc[ORDEM_AREAS]

ROTULOS_AREA = {
    "mainback": "Mainback", "coil_temps": "Coil Temps",
    "main_vent_sys": "Main Vent Sys", "ct_knobs": "CT Knobs",
    "ipc_comp": "IPC Comp", "ipc_cooling": "IPC Cooling",
    "oven_hmi": "Oven HMI",
}
rotulos_x = [ROTULOS_AREA[a] for a in ORDEM_AREAS]

METRICAS_LOGO = ["MAE_logo", "R2_logo_mediana", "dentro_de_10%_logo"]
TITULOS = ["MAE (\u00b0C)\n(menor \u00e9 melhor)",
           "R\u00b2, mediana entre sensores\n(maior \u00e9 melhor)",
           "Dentro de 10% (%)\n(maior \u00e9 melhor)"]

figura, eixos = plt.subplots(1, 3, figsize=(16, 5))

for eixo, metrica, titulo in zip(eixos, METRICAS_LOGO, TITULOS):
    valores = tabela_ord[metrica].values
    if metrica == "dentro_de_10%_logo":
        valores = valores * 100
    cores = ["#0072B2" if v >= 0 else "#D55E00" for v in valores]
    barras = eixo.bar(range(len(rotulos_x)), valores, color=cores,
                      edgecolor="white", linewidth=1.0)
    for barra, valor in zip(barras, valores):
        eixo.text(barra.get_x() + barra.get_width() / 2,
                  barra.get_height() + (abs(valores).max() * 0.02 if valor >= 0 else -abs(valores).max() * 0.06),
                  f"{valor:.2f}", ha="center",
                  va="bottom" if valor >= 0 else "top", fontsize=9)
    eixo.set_xticks(range(len(rotulos_x)))
    eixo.set_xticklabels(rotulos_x, rotation=35, ha="right", fontsize=9.5)
    eixo.set_title(titulo, fontsize=11)
    eixo.grid(axis="y", alpha=0.25)
    eixo.set_axisbelow(True)
    eixo.axhline(0, color="#555555", linewidth=0.8)
    for lado in ("top", "right"):
        eixo.spines[lado].set_visible(False)

figura.suptitle("M\u00e9tricas da valida\u00e7\u00e3o LOGO por \u00e1rea \u2014 Gradient Boosting",
                fontsize=13, y=1.03)
figura.tight_layout()

CAMINHO_LOGO_BARRAS = "../assets/graficos/modelo_candidato_logo_metricas.png"
figura.savefig(CAMINHO_LOGO_BARRAS, dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

print(f"Figura salva em: {CAMINHO_LOGO_BARRAS}")


### 8.2. Confirmação dos hiperparâmetros por busca em grade avaliada por LOGO

O `GridSearchCV` das Células 5 e 7 escolhe hiperparâmetros por validação cruzada de
3 folds **dentro do treino** do holdout — ~22 linhas por área, o que deixa ~7 linhas
por fold. `PARAMS_LOGO`, usado na avaliação oficial, foi definido nos testes do grupo.
Esta seção liga as duas coisas: roda a **mesma grade** (`GRADE_HIPERPARAMETROS`, 24
combinações) com o próprio LOGO como validação cruzada do `GridSearchCV`, e mostra
em que posição `PARAMS_LOGO` fica.

Como `PARAMS_LOGO` é o mesmo para as sete áreas, o ranking é feito pelo MAE LOGO
**médio entre as áreas**. A tabela por área mostra, para referência, a melhor
combinação de cada uma e quanto `PARAMS_LOGO` fica atrás dela.

**Leitura do resultado.** `PARAMS_LOGO` fica em **3º de 24**, a 0,05 °C (~1%) da
primeira colocada (`learning_rate = 0,05`, `subsample = 1,0`). As sete primeiras
combinações usam `max_depth = 3`, e as quatro primeiras usam `n_estimators = 100` com
`learning_rate` entre 0,03 e 0,05: o topo do ranking é um empate técnico, e a busca
confirma a região de hiperparâmetros escolhida pelo grupo. Trocar para a primeira
colocada não se justifica: a diferença é menor que a variação entre áreas, e escolher
a vencedora pelo mesmo LOGO que depois é reportado deixaria a métrica otimista.
`PARAMS_LOGO` segue como configuração oficial. O MAE de `PARAMS_LOGO` nesta busca é
idêntico ao da Célula 8, o que confirma que as duas avaliações medem a mesma coisa.

**Custo:** é a célula mais lenta do notebook — 24 combinações × 32 rodadas × um GBR
por termopar nas sete áreas, cerca de 14 minutos.

In [ ]:
# ============================================================
# CÉLULA 8.2 — BUSCA EM GRADE AVALIADA POR LOGO
# Mesma grade da Célula 5 (GRADE_HIPERPARAMETROS), agora com o LOGO como
# validação cruzada do GridSearchCV. Cada combinação é avaliada pelo MAE
# médio das 32 rodadas LOGO de cada área; o ranking final é a média entre as
# sete áreas, porque PARAMS_LOGO é único para todas elas.
# ============================================================
def grupos_por_configuracao(X_area):
    """Um grupo por configuração de ensaio (linhas idênticas em X)."""
    return pd.factorize(X_area.astype(str).agg("|".join, axis=1))[0]


def busca_em_grade_logo(caminho):
    """GridSearchCV com LeaveOneGroupOut; devolve o MAE LOGO de cada combinação."""
    X_area, Y_area, _, _, _ = preparar_area(caminho)
    busca = GridSearchCV(
        estimator=MultiOutputRegressor(
            GradientBoostingRegressor(random_state=SEMENTE), n_jobs=1
        ),
        param_grid=GRADE_HIPERPARAMETROS,
        cv=LeaveOneGroupOut(),
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
        refit=False,
    ).fit(X_area, Y_area, groups=grupos_por_configuracao(X_area))

    resultados = pd.DataFrame(busca.cv_results_["params"])
    resultados.columns = [c.replace("estimator__", "") for c in resultados.columns]
    resultados["MAE_logo"] = -busca.cv_results_["mean_test_score"]
    resultados["setor"] = caminho.split("/")[-1].replace("_data.csv", "")
    return resultados


HIPERPARAMETROS = ["n_estimators", "learning_rate", "max_depth", "subsample"]

busca_logo = pd.concat([busca_em_grade_logo(c) for c in PLANILHAS], ignore_index=True)

ranking_logo = (busca_logo.groupby(HIPERPARAMETROS)["MAE_logo"].mean()
                          .sort_values().reset_index())
ranking_logo.insert(0, "posicao", range(1, len(ranking_logo) + 1))

e_oficial = np.all([ranking_logo[p] == PARAMS_LOGO[p] for p in HIPERPARAMETROS], axis=0)
posicao_oficial = int(ranking_logo.loc[e_oficial, "posicao"].iloc[0])
mae_oficial = ranking_logo.loc[e_oficial, "MAE_logo"].iloc[0]
mae_melhor = ranking_logo["MAE_logo"].iloc[0]

print("=" * 90)
print(f"BUSCA EM GRADE AVALIADA POR LOGO — {len(ranking_logo)} combinações, "
      f"MAE médio entre as sete áreas")
print("=" * 90)
print(ranking_logo.round(4).to_string(index=False))

print(f"\nPARAMS_LOGO (oficial): posição {posicao_oficial} de {len(ranking_logo)}, "
      f"MAE = {mae_oficial:.4f} °C")
print(f"Melhor combinação:     MAE = {mae_melhor:.4f} °C "
      f"(diferença de {mae_oficial - mae_melhor:+.4f} °C)")

melhor_por_area = busca_logo.loc[busca_logo.groupby("setor", sort=False)["MAE_logo"].idxmin()]
oficial_por_area = busca_logo[
    np.all([busca_logo[p] == PARAMS_LOGO[p] for p in HIPERPARAMETROS], axis=0)
].set_index("setor")["MAE_logo"]
melhor_por_area = melhor_por_area.set_index("setor")
melhor_por_area["MAE_oficial"] = oficial_por_area
melhor_por_area["diferenca"] = melhor_por_area["MAE_oficial"] - melhor_por_area["MAE_logo"]

print("\nMelhor combinação de cada área x PARAMS_LOGO:")
print(melhor_por_area.round(4).to_string())

## 9. Explicabilidade do Gradient Boosting

### 9.1. Importância por impureza

O `MultiOutputRegressor` treina um regressor por termopar, então cada um tem sua
própria importância de variáveis — informação que os notebooks de Random Forest e
Decision Tree não produzem, porque ali um único modelo cobre todos os alvos da área.

A célula abaixo consolida essa matriz em duas leituras: a importância **média** entre
os termopares da área, e o termopar em que cada variável mais pesa. Os valores vêm
direto do modelo da Célula 5 (área `mainback`): `importancia` e
`matriz_importancia`.

**Ressalva de leitura:** pela H2 (Seção 4.2.4), as sete configurações estão
confundidas no desenho experimental — `losses` é redundante com `deflector_hole`.
Modelos de árvore distribuem o crédito preditivo de forma arbitrária entre variáveis
redundantes, então esta figura descreve **como o modelo decide**, não **como o fogão
aquece**. O título do gráfico carrega esse aviso.

Gera a Figura 19 da documentação.

In [ ]:
# ------------------------------------------------------------
# Figura 19 — explicabilidade do modelo candidato
# ------------------------------------------------------------
# Importâncias médias entre os termopares de `mainback` (Célula 5) e o
# termopar em que cada variável mais pesa — a leitura por sensor só existe
# porque o MultiOutputRegressor treina um GBR por termopar.
IMPORTANCIA_MAINBACK = {
    variavel: (importancia[variavel],
               matriz_importancia[variavel].idxmax(),
               matriz_importancia[variavel].max())
    for variavel in importancia.index
}


def grafico_explicabilidade(importancias, caminho):
    variaveis = list(importancias)[::-1]
    medias_imp = [importancias[v][0] for v in variaveis]
    sensores = [importancias[v][1] for v in variaveis]

    figura, eixo = plt.subplots(figsize=(13, 5.6))
    barras = eixo.barh(variaveis, medias_imp, color="#0072B2",
                       edgecolor="white", linewidth=1.0, height=0.68)
    barras[-1].set_color("#04456F")

    for barra, valor, sensor in zip(barras, medias_imp, sensores):
        eixo.text(valor + 0.008, barra.get_y() + barra.get_height() / 2,
                  f"{valor:.3f}", va="center", fontsize=10)
        eixo.text(valor + max(medias_imp) * 0.17, barra.get_y() + barra.get_height() / 2,
                  f"mais sensível: {sensor}", va="center", fontsize=9,
                  color="#4A4A4A", style="italic")

    eixo.set_xlim(0, max(medias_imp) * 2.2)
    eixo.set_xlabel("Importância média entre os dez termopares da área (mainback)",
                    fontsize=10.5)
    eixo.set_title("Explicabilidade — Gradient Boosting, área "
                   "Mainback\nA importância mede uso preditivo, não causa "
                   "(ver H2, Seção 4.2.4)", fontsize=12, pad=12)
    eixo.grid(axis="x", alpha=0.25)
    eixo.set_axisbelow(True)
    for lado in ("top", "right", "left"):
        eixo.spines[lado].set_visible(False)
    eixo.tick_params(axis="y", length=0, labelsize=10.5)

    figura.savefig(caminho, dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()
    return caminho


CAMINHO_EXPLICABILIDADE = "../assets/graficos/modelo_candidato_explicabilidade.png"

grafico_explicabilidade(IMPORTANCIA_MAINBACK, CAMINHO_EXPLICABILIDADE)
print()
print(f"Figura salva em: {CAMINHO_EXPLICABILIDADE}")

### 9.2. Validação da explicabilidade com SHAP

A importância por impureza (Seção 9.1) tem duas limitações: é adimensional — compara
variáveis entre si, mas não diz "em quantos graus" cada uma pesa — e, como qualquer
importância de árvore, distribui crédito de forma arbitrária entre variáveis
correlacionadas (H2, Seção 4.2.4). Esta seção recalcula a mesma leitura com **SHAP**
(SHapley Additive exPlanations), aplicado via `TreeExplainer` a cada um dos dez
`GradientBoostingRegressor` de `mainback` (Célula 5).

A vantagem é dupla: o valor SHAP tem a mesma unidade do alvo (°C) — dá para dizer
"`blower_direction` desloca a previsão em ~3 °C, em média", não só "pesa 0,29" — e o
método é aditivo por construção: a soma das contribuições de uma linha bate
exatamente com a diferença entre a predição e a média do modelo, sem depender de
como a árvore particiona o espaço internamente.

A Figura 20 mostra o resultado, na mesma agregação da Figura 19: média de |SHAP|
entre os dez termopares, com o termopar mais sensível a cada variável.


In [ ]:
# ------------------------------------------------------------
# Figura 20 -- validacao da explicabilidade com SHAP
# ------------------------------------------------------------
# TreeExplainer roda por termopar (cada GBR do MultiOutputRegressor e
# independente) sobre as 32 linhas de `mainback`; agrega |SHAP| por coluna
# one-hot de volta a variavel original (mesma logica de `importancias_por_alvo`)
# e depois tira a media entre os dez termopares -- mesma leitura da Figura 19,
# mas em graus Celsius em vez de importancia adimensional.
import shap

def importancia_shap_por_alvo(modelo_multioutput, X, independentes, alvos):
    """Matriz termopar x variavel com a media de |SHAP| por termopar,
    somando as colunas one-hot de volta a variavel original."""
    shap_por_alvo = {}
    for alvo, estimador in zip(alvos, modelo_multioutput.estimators_):
        valores_shap = shap.TreeExplainer(estimador).shap_values(X)
        shap_por_alvo[alvo] = np.abs(valores_shap).mean(axis=0)

    bruta = pd.DataFrame(shap_por_alvo, index=X.columns).T

    def variavel_original(coluna):
        candidatos = [v for v in independentes if coluna.startswith(v + "_")]
        return max(candidatos, key=len)

    mapa_variavel = {coluna: variavel_original(coluna) for coluna in X.columns}
    return bruta.T.groupby(mapa_variavel).sum().T


matriz_shap_mainback = importancia_shap_por_alvo(
    modelo, X, VARIAVEIS_INDEPENDENTES, VARIAVEIS_ALVO
)
shap_mainback = matriz_shap_mainback.mean().sort_values(ascending=False)

print("Importancia SHAP media (|valor SHAP| em graus C, entre os 10 termopares):")
print(shap_mainback.round(4).to_string())

print("\nTermopar mais sensivel a cada variavel (SHAP):")
for variavel in shap_mainback.index:
    alvo = matriz_shap_mainback[variavel].idxmax()
    print(f"   {variavel:24s} -> {alvo} ({matriz_shap_mainback.loc[alvo, variavel]:.3f})")

comparacao_shap_impureza = pd.DataFrame({
    "impureza": importancia,
    "SHAP (graus C)": shap_mainback,
}).sort_values("SHAP (graus C)", ascending=False)
comparacao_shap_impureza["rank impureza"] = comparacao_shap_impureza["impureza"].rank(ascending=False).astype(int)
comparacao_shap_impureza["rank SHAP"] = comparacao_shap_impureza["SHAP (graus C)"].rank(ascending=False).astype(int)

print("\nComparacao impureza (Figura 19) x SHAP (Figura 20):")
print(comparacao_shap_impureza.round(4).to_string())


def grafico_shap(shap_medio, matriz_shap, caminho):
    variaveis = list(shap_medio.index)[::-1]
    valores = [shap_medio[v] for v in variaveis]
    sensores = [matriz_shap[v].idxmax() for v in variaveis]

    figura, eixo = plt.subplots(figsize=(13, 5.6))
    barras = eixo.barh(variaveis, valores, color="#0072B2",
                       edgecolor="white", linewidth=1.0, height=0.68)
    barras[-1].set_color("#04456F")

    for barra, valor, sensor in zip(barras, valores, sensores):
        eixo.text(valor + 0.08, barra.get_y() + barra.get_height() / 2,
                  f"{valor:.2f} \u00b0C", va="center", fontsize=10)
        eixo.text(valor + 0.65, barra.get_y() + barra.get_height() / 2,
                  f"mais sensivel: {sensor}", va="center", fontsize=9,
                  color="#4A4A4A", style="italic")

    eixo.set_xlim(0, max(valores) * 1.55)
    eixo.set_xlabel("|valor SHAP| medio entre os dez termopares da area (mainback), em \u00b0C",
                    fontsize=10.5)
    eixo.set_title("Valida\u00e7\u00e3o da explicabilidade com SHAP \u2014 Gradient Boosting, \u00e1rea "
                   "Mainback\nContribui\u00e7\u00e3o m\u00e9dia de cada vari\u00e1vel para a previs\u00e3o, em \u00b0C",
                   fontsize=12, pad=12)
    eixo.grid(axis="x", alpha=0.25)
    eixo.set_axisbelow(True)
    for lado in ("top", "right", "left"):
        eixo.spines[lado].set_visible(False)
    eixo.tick_params(axis="y", length=0, labelsize=10.5)

    figura.savefig(caminho, dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()
    return caminho


CAMINHO_SHAP = "../assets/graficos/modelo_candidato_shap.png"

grafico_shap(shap_mainback, matriz_shap_mainback, CAMINHO_SHAP)
print()
print(f"Figura salva em: {CAMINHO_SHAP}")


### 9.3. Explicabilidade SHAP nas sete áreas

A Seção 9.2 aplicou SHAP só em `mainback`, a área analisada em detalhe. Esta
seção estende a mesma leitura às sete áreas, reaproveitando os modelos já
treinados na Célula 7 — sem retreinar nada.

Como cada área opera em um patamar de temperatura diferente (de ~47 °C em
`CT Knobs` a ~124 °C em `Coil Temps`), o valor SHAP em °C não é diretamente
comparável entre áreas: uma variável "vale" mais graus numa área que já é
mais quente, mesmo que pese proporcionalmente menos. Por isso a Figura 23
normaliza cada linha para somar 100%, comparando a **participação relativa**
de cada variável dentro de cada área, no mesmo formato de mapa de calor usado
para o Random Forest (`Random_for_regre.ipynb`).


In [ ]:
# ------------------------------------------------------------
# Figura 23 -- mapa de calor: participacao da importancia SHAP por area
# ------------------------------------------------------------
# Estende a Secao 9.2 (que olhava so `mainback`) para as sete areas: reusa o
# modelo ja treinado de cada setor (Celula 7) e recalcula a importancia SHAP
# com o mesmo `TreeExplainer` por termopar, normalizando cada linha para
# somar 100% -- assim a comparacao entre areas e sobre PARTICIPACAO relativa,
# nao sobre graus C absolutos (que dependem do patamar termico de cada area).
import shap

IMPORTANCIA_SHAP_POR_AREA = {}

for caminho in PLANILHAS:
    nome_area = caminho.split("/")[-1].replace("_data.csv", "")
    X_area, Y_area, alvos_area, dicionario_area, _ = preparar_area(caminho)
    modelo_area = MODELOS_POR_AREA[nome_area]["modelo"]

    matriz_shap_area = importancia_shap_por_alvo(
        modelo_area, X_area, VARIAVEIS_INDEPENDENTES, alvos_area
    )
    IMPORTANCIA_SHAP_POR_AREA[nome_area] = matriz_shap_area.mean()
    print(f"OK   {nome_area:15s} (colunas one-hot: {X_area.shape[1]}, alvos: {len(alvos_area)})")

tabela_shap_areas = pd.DataFrame(IMPORTANCIA_SHAP_POR_AREA).T

print("\n" + "=" * 100)
print("IMPORTANCIA SHAP GLOBAL (|SHAP| medio em graus C, entre os termopares de cada area)")
print("=" * 100)
print(tabela_shap_areas.round(3).to_string())

tabela_shap_norm = tabela_shap_areas.div(tabela_shap_areas.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(9.5, 6))
im = ax.imshow(tabela_shap_norm.values, cmap="YlOrRd", aspect="auto", vmin=0)

ax.set_xticks(range(len(tabela_shap_norm.columns)))
ax.set_xticklabels(tabela_shap_norm.columns, rotation=35, ha="right")
ax.set_yticks(range(len(tabela_shap_norm.index)))
ax.set_yticklabels(tabela_shap_norm.index)

limite_cor = tabela_shap_norm.values.max()
for i in range(tabela_shap_norm.shape[0]):
    for j in range(tabela_shap_norm.shape[1]):
        valor = tabela_shap_norm.values[i, j]
        cor_texto = "white" if valor > limite_cor * 0.6 else "#333333"
        ax.text(j, i, f"{valor:.0%}", ha="center", va="center", fontsize=8, color=cor_texto)

ax.set_title("Participa\u00e7\u00e3o de cada vari\u00e1vel na import\u00e2ncia SHAP, por \u00e1rea\n(cada linha soma 100%) \u2014 Gradient Boosting",
             fontsize=11, fontweight="bold")
fig.colorbar(im, ax=ax, label="participa\u00e7\u00e3o na import\u00e2ncia total da \u00e1rea")
plt.tight_layout()

CAMINHO_SHAP_HEATMAP = "../assets/graficos/modelo_candidato_shap_por_area.png"
fig.savefig(CAMINHO_SHAP_HEATMAP, dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

print("\nVariavel mais importante por area:")
print(tabela_shap_norm.idxmax(axis=1).to_string())
print(f"\nFigura salva em: {CAMINHO_SHAP_HEATMAP}")
